# 🎯 05-2. 통합 시스템 모델 비교 (Unified System Comparison)

이 노트북은 **분류(Classification) + 검색(RAG) + 생성(Generation)**이 결합된 최종 법률 AI 시스템에서, **다양한 LLM 모델(Qwen, Llama, Mistral 등)**이 실제로 어떤 차이를 보이지는 비교 실험하는 단계입니다.

### 🧪 실험 목표
1.  **동일한 입력**에 대해 각 모델이 어떻게 반응하는지 비교
2.  **한국어 자연스러움**, **법적 논리**, **친절도** 평가
3.  최종 서비스에 사용할 **Best Model 선정**

## 1. 환경 설정

In [3]:
# 필요 패키지 설치
!pip install transformers torch pandas langchain-ollama langchain-community langchain-huggingface faiss-cpu -q

# Ollama 설치 및 실행 (Colab용)
import os
if os.system("ollama --version") != 0:
    print("⚙️ Ollama 설치 중...")
    !sudo apt-get install -y pciutils lshw zstd
    !curl -fsSL https://ollama.com/install.sh | sh

import torch
import pandas as pd
import pickle
import time
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ 실행 디바이스: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
⚙️ Ollama 설치 중...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 41 not

## 2. 모델 다운로드 (비교군)
다양한 성향의 모델을 준비합니다.

In [4]:
# 백그라운드에서 Ollama 서버 실행
get_ipython().system_raw('ollama serve &')
time.sleep(5)

# 비교할 모델 리스트
CANDIDATE_MODELS = [
    "qwen2.5:7b",    # (추천) 한국어/다국어 성능 우수
    "llama3:8b",     # (기준) 범용적인 고성능 모델
    "mistral",       # (논리) 논리적 추론 강점
    "gemma:2b"       # (속도) 매우 빠른 속도
]

for model in CANDIDATE_MODELS:
    print(f"📥 {model} 다운로드 중...")
    !ollama pull {model}
    print(f"✅ {model} 준비 완료")

📥 qwen2.5:7b 다운로드 중...

✅ qwen2.5:7b 준비 완료
📥 llama3:8b 다운로드 중...

✅ llama3:8b 준비 완료
📥 mistral 다운로드 중...

✅ mistral 준비 완료
📥 gemma:2b 다운로드 중...

✅ gemma:2b 준비 완료


## 3. 리소스 로드 (Drive 연동)

In [5]:
# 구글 드라이브 마운트
from google.colab import drive
if not os.path.exists('/content/drive'):
    print("📂 구글 드라이브 마운트 중...")
    drive.mount('/content/drive')

# 경로 설정 (사용자 지정 경로: Colab Notebooks)
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks"
MODEL_PATH = f"{BASE_DIR}/best_model/final"
VECTOR_DB_PATH = f"{BASE_DIR}/vector_store" # (참고) 벡터 DB가 있다면 사용

# 1. 분류 모델 로드
try:
    print(f"🔄 분류 모델 로드: {MODEL_PATH}")
    # 토크나이저 & 모델
    try:
        clf_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
        clf_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)
    except:
        print("⚠️ 로컬 로드 실패, 기본 설정으로 시도...")
        clf_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
        clf_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

    clf_model.to(device)
    clf_model.eval()

    # 라벨 인코더
    with open(f"{MODEL_PATH}/label_encoder.pkl", "rb") as f:
        label_encoder = pickle.load(f)
    print("✅ 분류 모델 로드 성공")
except Exception as e:
    print(f"❌ 분류 모델 로드 실패: {e}")
    print("⚠️ 03-4 노트북을 먼저 실행해주세요.")
    raise e

# 2. 벡터 스토어 (RAG) 로드 - 없으면 Mock(가짜) 데이터 사용
rag_enabled = False
try:
    if os.path.exists(VECTOR_DB_PATH):
        print(f"🔄 벡터 스토어 로드: {VECTOR_DB_PATH}")
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vectorstore = FAISS.load_local(VECTOR_DB_PATH, embeddings, allow_dangerous_deserialization=True)
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        rag_enabled = True
        print("✅ 벡터 스토어 로드 성공")
    else:
        print("⚠️ 벡터 스토어가 없습니다. (Mock 모드로 동작)")
except Exception as e:
    print(f"⚠️ 벡터 스토어 로드 중 오류: {e}")

📂 구글 드라이브 마운트 중...
Mounted at /content/drive
🔄 분류 모델 로드: /content/drive/MyDrive/Colab Notebooks/best_model/final
✅ 분류 모델 로드 성공
⚠️ 벡터 스토어가 없습니다. (Mock 모드로 동작)


## 4. 통합 AI 클래스 정의

In [6]:
import json

class UnifiedLegalAI:
    def __init__(self, classifier, tokenizer, label_encoder, retriever, llm_model_name):
        self.classifier = classifier
        self.tokenizer = tokenizer
        self.label_encoder = label_encoder
        self.retriever = retriever
        self.llm = ChatOllama(model=llm_model_name)

    def classify_dispute(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.classifier(**inputs)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        predicted_class_idx = predictions.item()
        predicted_category = self.label_encoder.inverse_transform([predicted_class_idx])[0]
        return predicted_category

    def retrieve_precedents(self, query, category=None):
        if not self.retriever:
            # Mock data for RAG if not enabled
            return [
                "임대차 분쟁은 계약 관계에 따라 보증금 반환, 차임 지급, 계약 해지 여부가 주요 쟁점이 된다.",
                "계약 위반에 따른 손해배상 책임은 채무 불이행 여부와 손해 발생의 인과관계를 기준으로 판단된다.",
                "사기죄는 단순한 채무 불이행이 아니라 고의적인 기망행위가 있었는지가 핵심 판단 요소이다."
            ]
        # 실제 RAG 구현 (여기서는 간략화)
        docs = self.retriever.get_relevant_documents(query)
        # category가 주어졌을 경우 관련 판례만 필터링하는 로직 추가 가능
        return [doc.page_content for doc in docs]

    def generate_explanation(self, text, category, precedents):
        try:
            context_str = "\n".join([f"- {p}" for p in precedents])

            prompt = f"""
[역할]
당신은 한국의 법률 분쟁유형을 설명하는 AI입니다.
반드시 한국어로만 작성하세요.

[언어 규칙 - 매우 중요]
- 중국어, 영어, 한자, 일본어가 포함되면 오류입니다.
- 반드시 한국어 문장만 사용하세요.

[입력 정보]
- 사건 내용: {text}
- 분쟁 유형: {category}
- 참고 법령/판례:
{context_str}

[중요한 제한]
- 법률 조언을 하지 마세요.
- 소송, 고소, 합의, 대응 방법을 제시하지 마세요.
- 판단 근거 설명만 하세요.

[출력 목적]
아래 UI에 표시할 정보를 JSON 형식으로 생성하세요.

[JSON 출력 형식]
{{
  "main_category": "민사/형사/행정/가사/노동 중 1개",
  "sub_category": "세부 분류",
  "title": "사건을 한 문장으로 요약",
  "tags": ["", "", ""],
  "legal_judgment": "분쟁 유형 판단 근거 (250~350자)",
  "related_laws": ["", ""]
}}

[출력 규칙]
- 반드시 JSON만 출력하세요.
- JSON 외 텍스트 출력 금지
"""

            start_time = time.time()
            response = self.llm.invoke(prompt)
            duration = time.time() - start_time

            raw = response.content.strip()
            raw = raw[raw.find("{"): raw.rfind("}") + 1]
            data = json.loads(raw)

            return data, duration

        except Exception as e:
            print(f"생성 오류 발생: {e}")
            return {"error": str(e)}, 0

    def run(self, query):
        start_full_process_time = time.time()

        # 1. 분류 (Classification)
        category = self.classify_dispute(query)

        # 2. 검색 (RAG)
        precedents = self.retrieve_precedents(query, category)

        # 3. 생성 (Generation)
        advice_data, generation_duration = self.generate_explanation(query, category, precedents)

        end_full_process_time = time.time()
        full_duration = end_full_process_time - start_full_process_time

        return {
            "model": self.llm.model,
            "category": category,
            "precedents": precedents,
            "advice": advice_data,
            "duration": full_duration,
            "generation_duration": generation_duration
        }

## 5. 비교 실험 실행

In [7]:
test_input = "인테리어 업체와 3천만원 공사 계약을 맺고 선금 50%를 줬는데, 철거만 해놓고 연락이 두절되었습니다. 사기죄 고소가 가능한가요?"

print("🧪 모델 비교 실험 시작...")
print(f"📌 입력 질문: {test_input}\n")

results = []

for model_name in CANDIDATE_MODELS:
    print(f"🔄 [{model_name}] 실행 중...", end=" ")

    # 시스템 인스턴스 생성
    ai_system = UnifiedLegalAI(
        classifier=clf_model,
        tokenizer=clf_tokenizer,
        label_encoder=label_encoder,
        retriever=retriever if rag_enabled else None,
        llm_model_name=model_name
    )

    # 실행
    res = ai_system.run(test_input)
    results.append(res)
    print(f"완료 ({res['duration']:.2f}초)")

print("\n✅ 모든 모델 테스트 완료!")

🧪 모델 비교 실험 시작...
📌 입력 질문: 인테리어 업체와 3천만원 공사 계약을 맺고 선금 50%를 줬는데, 철거만 해놓고 연락이 두절되었습니다. 사기죄 고소가 가능한가요?

🔄 [qwen2.5:7b] 실행 중... 완료 (36.94초)
🔄 [llama3:8b] 실행 중... 완료 (29.16초)
🔄 [mistral] 실행 중... 완료 (34.76초)
🔄 [gemma:2b] 실행 중... 완료 (11.73초)

✅ 모든 모델 테스트 완료!


## 6. 결과 비교 (Side-by-Side)

In [8]:
for i, res in enumerate(results, start=1):
    print("=" * 80)
    print(f" Model {i}: {res['model']} (시간: {res['duration']:.2f}s)")
    print(f" 분류: {res['category']}")
    print("-" * 80)

    advice = res["advice"]

    print(" 사건 요약")
    print(f"  ▸ 대분류     : {advice.get('main_category')}")
    print(f"  ▸ 소분류     : {advice.get('sub_category')}")
    print(f"  ▸ 제목       : {advice.get('title')}")
    print(f"  ▸ 태그       : {', '.join(advice.get('tags', []))}")

    print("\n 법률 판단")
    print(advice.get("legal_judgment"))

    print("\n 관련 법령")
    for law in advice.get("related_laws", []):
        print(f"  - {law}")

    print("=" * 80)
    print()


 Model 1: qwen2.5:7b (시간: 36.94s)
 분류: 민사
--------------------------------------------------------------------------------
 사건 요약
  ▸ 대분류     : 민사
  ▸ 소분류     : 계약 위반 및 손해배상
  ▸ 제목       : 인테리어 업체와의 공사 계약을 체결했으나 선금 지급 후 연락이 두절되어 사기죄 고소가 가능한지
  ▸ 태그       : 계약, 선금, 연락두절, 사기

 법률 판단
이 사건은 민사 분쟁으로 분류되며, 선금을 지급하고 공사가 진행되지 않은 상태에서 연락이 두절된 경우 사기죄 고소 여부는 채무 불이행 자체보다는 업체의 기망행위 유무와 그 행위로 인해 발생한 손해를 판단하는 것이 핵심입니다. 그러나 이 점을 입증하려면 충분한 증거가 필요하며, 법률적 판단은 법원에서 이루어집니다.

 관련 법령
  - 민법
  - 형사법

 Model 2: llama3:8b (시간: 29.16s)
 분류: 민사
--------------------------------------------------------------------------------
 사건 요약
  ▸ 대분류     : 민사
  ▸ 소분류     : 계약 위반/손해배상
  ▸ 제목       : 인테리어 업체와 3천만원 공사 계약의 철거만 해놓고 연락이 두절된 경우
  ▸ 태그       : 계약 위반, 손해배상, 사기죄

 법률 판단
해당 사건은 민사 분쟁 유형에 해당한다. 인테리어 업체가 철거만 해놓고 연락이 두절되자, 계약위반에 따른 손해배상 책임을 논의해야 한다. 사기죄 고소 여부를 결정하기 위해서는 고의적인 기망행위 여부를 판단해야 하므로, 추가적으로 조사가 필요하다.

 관련 법령
  - 계약법
  - 손해배상법

 Model 3: mistral (시간: 34.76s)
 분류: 민사
------------------------------------------------

## 7. 정량적 요약

In [9]:
summary_df = pd.DataFrame(results)
summary_df['length'] = summary_df['advice'].apply(len)
print("📊 모델별 성능 요약")
display(summary_df[['model', 'duration', 'length', 'category']])

# 추천 알고리즘 (간단 버전)
best_row = summary_df.sort_values(by='duration').iloc[0] # 가장 빠른 모델
print(f"\n💡 [속도 추천]: {best_row['model']}")

# 한국어(가,힣) 비중이 높은 모델
def korean_score(text):
    if not isinstance(text, str):
        return 0.0 # Handle non-string input gracefully
    if len(text) == 0:
        return 0.0
    return sum(1 for c in text if ord('가') <= ord(c) <= ord('힣')) / len(text)

# 'advice' 컬럼의 딕셔너리에서 'legal_judgment' 필드를 추출하여 korean_score 적용
summary_df['ko_score'] = summary_df['advice'].apply(lambda x: korean_score(x.get('legal_judgment', '')))
best_ko = summary_df.sort_values(by='ko_score', ascending=False).iloc[0]
print(f"💡 [한국어 품질 추천]: {best_ko['model']}")

📊 모델별 성능 요약


,model,duration,length,category
0,qwen2.5:7b,36.938377,6,민사
1,llama3:8b,29.158234,6,민사
2,mistral,34.760620,6,민사
3,gemma:2b,11.728144,6,민사



💡 [속도 추천]: gemma:2b
💡 [한국어 품질 추천]: llama3:8b
